In [3]:
import os
import json
import re
import pandas as pd
import mediapipe as mp #0.10.9
from moviepy import VideoFileClip
import cv2

# Face copping:

In [5]:
import matplotlib.pyplot as plt

# Initialize ONCE (outside the function)
mp_face_detection = mp.solutions.face_detection
face_detector = mp_face_detection.FaceDetection(
    model_selection=0,
    min_detection_confidence=0.5
)

def facial_cropping(img, face_detector):
    h, w, _ = img.shape

    results = face_detector.process(img)

    if results.detections:
        # Pick highest confidence detection
        detection = max(results.detections, key=lambda d: d.score[0])
        bbox = detection.location_data.relative_bounding_box

        # Convert relative → absolute coordinates
        x = int(bbox.xmin * w)
        y = int(bbox.ymin * h)
        box_w = int(bbox.width * w)
        box_h = int(bbox.height * h)

        # Clamp negative values (important!)
        x = max(0, x)
        y = max(0, y)

        # Add 30% padding (based on face width)
        padding = max(int(box_w * 0.5), int(box_h * 0.5))

        x1 = max(0, x - padding)
        y1 = max(0, y - padding)
        x2 = min(w, x + box_w + padding)
        y2 = min(h, y + box_h + padding)

        cropped_face = img[y1:y2, x1:x2]

        return cropped_face

    else:
        return img


I0000 00:00:1777035546.448898   14018 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1777035546.452190   14341 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, LLVM 20.1.2, DRM 3.64, 6.17.0-22-generic)


# Video frame sampling:

In [3]:
def process_video(input_dir, output_dir, window_size=5, fps=1, face_cropping = False):

    print(f'Processing: {input_dir}...')

    if not face_cropping:
        for subject in os.listdir(input_dir):
            for act in os.listdir(os.path.join(input_dir, subject)):
                video_path = os.path.join(input_dir, subject, act)
                clip = VideoFileClip(video_path)
                duration = clip.duration
                subject_act = os.path.basename(video_path).split('.')[0]
                
                # Calculate number of windows
                num_windows = int(duration // window_size)
                
                for i in range(num_windows):
                    start_time = i * window_size
                    end_time = (i + 1) * window_size
                    
                    # Create a subclip
                    subclip = clip.subclipped(start_time, end_time)
                    
                    # Save frames for this window
                    window_dir = os.path.join(output_dir, subject, subject_act, f"window_{i}")
                    os.makedirs(window_dir, exist_ok=True)
                    
                    # Extract frames
                    # formatted as frame_0.jpg, frame_1.jpg, etc.
                    subclip.write_images_sequence(os.path.join(window_dir, "frame_%d.jpg"), fps=fps, logger=None)
                
                clip.close()

            print(f'{subject} done.')
        


    else: #Face cropping using haar cascade
        mp_face_detection = mp.solutions.face_detection
        face_detector = mp_face_detection.FaceDetection(
            model_selection=0,
            min_detection_confidence=0.5
            )

        for subject in os.listdir(input_dir):
            for act in os.listdir(os.path.join(input_dir, subject)):
                video_path = os.path.join(input_dir, subject, act)
                clip = VideoFileClip(video_path)
                duration = clip.duration
                subject_act = os.path.basename(video_path).split('.')[0]
                
                num_windows = int(duration // window_size)

                for i in range(num_windows):
                    start_time = i * window_size
                    end_time = (i + 1) * window_size
                    
                    # Note: MoviePy uses 'subclip', not 'subclipped'
                    subclip = clip.subclipped(start_time, end_time) 
                    
                    window_dir = os.path.join(output_dir, subject, subject_act, f"window_{i}")
                    os.makedirs(window_dir, exist_ok=True)
                    
                    # 2. Iterate through the frames manually to intercept them
                    for frame_idx, frame in enumerate(subclip.iter_frames(fps=fps)):
                        
                        cropped_face = facial_cropping(frame, face_detector)
                        cropped_face_bgr = cv2.cvtColor(cropped_face, cv2.COLOR_RGB2BGR)
                        frame_path = os.path.join(window_dir, f"frame_{frame_idx}.jpg")
                        cv2.imwrite(frame_path, cropped_face_bgr)

                clip.close()
                
            print(f'{subject} done.')
    print('All done!')


In [ ]:
process_video("../data/original", "../data/5s1fps_cropped", window_size=5, fps=1, face_cropping=True)
process_video("../data/original", "../data/5s2fps_cropped", window_size=5, fps=2, face_cropping=True)
process_video("../data/original", "../data/5s5fps_cropped", window_size=5, fps=5, face_cropping=True)
process_video("../data/original", "../data/10s1fps_cropped", window_size=10, fps=1, face_cropping=True)
process_video("../data/original", "../data/10s2fps_cropped", window_size=10, fps=2, face_cropping=True)
process_video("../data/original", "../data/10s5fps_cropped", window_size=10, fps=5, face_cropping=True)

In [6]:
def process_video(
    input_dir,
    output_dir,
    window_size=5,
    fps=1,
    overlap=0.0,
    face_cropping=False
):
    """
    Processes videos into frame windows.

    overlap:
        0.0 = no overlap
        0.5 = 50% overlap
        0.75 = 75% overlap

    Example:
        window_size=10, overlap=0.5
        windows: [0-10], [5-15], [10-20], ...
    """

    if overlap < 0 or overlap >= 1:
        raise ValueError("overlap must be >= 0 and < 1")

    stride = window_size * (1 - overlap)

    print(f"Processing: {input_dir}...")
    print(f"Window size: {window_size}s | FPS: {fps} | Overlap: {overlap * 100:.0f}%")
    print(f"Stride: {stride:.2f}s")

    def get_windows(duration, window_size, stride):
        """
        Returns list of (window_id, start_time, end_time).
        Drops incomplete final windows, matching your original behavior.
        """
        windows = []
        start_time = 0.0
        window_id = 0

        while start_time + window_size <= duration:
            end_time = start_time + window_size
            windows.append((window_id, start_time, end_time))

            window_id += 1
            start_time += stride

        return windows

    if not face_cropping:

        for subject in sorted(os.listdir(input_dir)):
            subject_dir = os.path.join(input_dir, subject)

            if not os.path.isdir(subject_dir):
                continue

            for act in sorted(os.listdir(subject_dir)):
                video_path = os.path.join(subject_dir, act)

                if not os.path.isfile(video_path):
                    continue

                clip = VideoFileClip(video_path)
                duration = clip.duration
                subject_act = os.path.basename(video_path).split(".")[0]

                windows = get_windows(duration, window_size, stride)

                for window_id, start_time, end_time in windows:
                    subclip = clip.subclipped(start_time, end_time)

                    window_dir = os.path.join(
                        output_dir,
                        subject,
                        subject_act,
                        f"window_{window_id}"
                    )

                    os.makedirs(window_dir, exist_ok=True)

                    subclip.write_images_sequence(
                        os.path.join(window_dir, "frame_%d.jpg"),
                        fps=fps,
                        logger=None
                    )

                clip.close()

            print(f"{subject} done.")

    else:
        mp_face_detection = mp.solutions.face_detection
        face_detector = mp_face_detection.FaceDetection(
            model_selection=0,
            min_detection_confidence=0.5
        )

        for subject in sorted(os.listdir(input_dir)):
            subject_dir = os.path.join(input_dir, subject)

            if not os.path.isdir(subject_dir):
                continue

            for act in sorted(os.listdir(subject_dir)):
                video_path = os.path.join(subject_dir, act)

                if not os.path.isfile(video_path):
                    continue

                clip = VideoFileClip(video_path)
                duration = clip.duration
                subject_act = os.path.basename(video_path).split(".")[0]

                windows = get_windows(duration, window_size, stride)

                for window_id, start_time, end_time in windows:
                    subclip = clip.subclipped(start_time, end_time)

                    window_dir = os.path.join(
                        output_dir,
                        subject,
                        subject_act,
                        f"window_{window_id}"
                    )

                    os.makedirs(window_dir, exist_ok=True)

                    for frame_idx, frame in enumerate(subclip.iter_frames(fps=fps)):
                        cropped_face = facial_cropping(frame, face_detector)
                        cropped_face_bgr = cv2.cvtColor(cropped_face, cv2.COLOR_RGB2BGR)

                        frame_path = os.path.join(
                            window_dir,
                            f"frame_{frame_idx}.jpg"
                        )

                        cv2.imwrite(frame_path, cropped_face_bgr)

                clip.close()

            print(f"{subject} done.")

        face_detector.close()

    print("All done!")

In [9]:
process_video(input_dir="../data/original", output_dir="../data/5s5fps_20overlap_cropped", window_size=5, fps=5, overlap=0.2, face_cropping=True)

Processing: ../data/original...
Window size: 5s | FPS: 5 | Overlap: 20%
Stride: 4.00s


I0000 00:00:1777036495.281694   14018 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1777036495.283311  195710 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, LLVM 20.1.2, DRM 3.64, 6.17.0-22-generic)


2ea4 done.
2hpu done.
2z7d done.
45lx done.
4e8r done.
4woj done.
5f7t done.
6g6y done.
6k5f done.
71i5 done.
7h5u done.
8g4y done.
8i4i done.
9j3o done.
9t6n done.
9txq done.
a1k9 done.
b2l8 done.
b9w0 done.
bfl5 done.
c3m7 done.
chdf done.
ctzy done.
cxj0 done.
d4n6 done.
e5p4 done.
f6q3 done.
g7r2 done.
g9j5 done.
h7j3 done.
h8r2 done.
h8s1 done.
i9t9 done.
iqyg done.
j9h8 done.
k2v7 done.
k67g done.
kkf5 done.
kycf done.
m8g5 done.
p9i3 done.
qw5t done.
r3zm done.
r5s8 done.
t6v9 done.
tmvd done.
uymz done.
v8mh done.
w2t5 done.
wssm done.
x1q3 done.
y8c3 done.
y9z6 done.
All done!
